In [1]:
# multivariate lstm example
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy import array
from numpy import hstack
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Input
from tqdm import tqdm
import os
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
# Configure TensorFlow to use all available CPUs
tf.config.threading.set_intra_op_parallelism_threads(0)  # Use all CPU threads for operations
tf.config.threading.set_inter_op_parallelism_threads(0)  # Use all CPU threads for inter-operation parallelism

os.chdir('/tf/Capstone/')

2025-03-28 23:56:28.814955: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Import the relevant data from the database
def feature_data_retrieve():
    # Make the connection to the database
    conn = sqlite3.connect('data.sqlite')
    # Write the query
    query = '''SELECT station_id, docks_available, timestamp, year, month, day, hour, temperature_2m, apparent_temperature, relative_humidity_2m, wind_speed_10m, sunshine_duration, rain FROM data_table WHERE year IN (2022, 2023, 2024) AND month IN (1, 2, 3, 4, 5) AND station_id BETWEEN 1 AND 496'''
    # Read the query into a pandas dataframe
    for chunk in pd.read_sql_query(query, conn, chunksize=10000):
        yield pd.DataFrame(chunk)
    # Close the connection
    conn.close()

feature_data = pd.DataFrame()

# Load the data
for chunk in tqdm(feature_data_retrieve()):
    feature_data = pd.concat([feature_data, chunk])

# Convert the timestamp column to datetime
feature_data['timestamp'] = pd.to_datetime(feature_data['timestamp'])

# Make timestamp the index
feature_data.set_index('timestamp', inplace=True)

# Sort the feature data by station_id and then station_id by timestamp
feature_data.sort_values(by=['station_id', 'timestamp'], inplace=True)

# Engineer features, four columns which will have docks available shifted by 1, 2, 3, and 4 hours before the current timestamp. They are called ctx-1, ctx-2, ctx-3, and ctx-4
feature_data['ctx-1'] = feature_data.groupby('station_id')['docks_available'].shift(1)
feature_data['ctx-2'] = feature_data.groupby('station_id')['docks_available'].shift(2)
feature_data['ctx-3'] = feature_data.groupby('station_id')['docks_available'].shift(3)
feature_data['ctx-4'] = feature_data.groupby('station_id')['docks_available'].shift(4)


# Drop rows with missing values
feature_data.dropna(inplace=True)

# Drop columns that are not needed
feature_data.drop(columns=['relative_humidity_2m'], inplace=True)
feature_data.drop(columns=['temperature_2m'], inplace=True)

# Convert the timestamp into a column again and reset the index
feature_data.reset_index(inplace=True)



# Normalize the columns apparent_temperature, wind_speed_10m, sunshine_duration, and rain
feature_data['apparent_temperature'] = (feature_data['apparent_temperature'] - feature_data['apparent_temperature'].min()) / (feature_data['apparent_temperature'].max() - feature_data['apparent_temperature'].min())
feature_data['wind_speed_10m'] = (feature_data['wind_speed_10m'] - feature_data['wind_speed_10m'].min()) / (feature_data['wind_speed_10m'].max() - feature_data['wind_speed_10m'].min())
feature_data['sunshine_duration'] = (feature_data['sunshine_duration'] - feature_data['sunshine_duration'].min()) / (feature_data['sunshine_duration'].max() - feature_data['sunshine_duration'].min())
feature_data['rain'] = (feature_data['rain'] - feature_data['rain'].min()) / (feature_data['rain'].max() - feature_data['rain'].min())

display(feature_data.head(10))

3it [00:00, 25.76it/s]

514it [01:28,  5.83it/s]


,timestamp,station_id,docks_available,year,month,day,hour,apparent_temperature,wind_speed_10m,sunshine_duration,rain,ctx-1,ctx-2,ctx-3,ctx-4
0,2022-01-01 04:00:00,1,0.717391,2022,1,1,4,0.262114,0.276638,0.000000,0.0,0.717391,0.737319,0.726449,0.682971
1,2022-01-01 05:00:00,1,0.672101,2022,1,1,5,0.263686,0.294422,0.000000,0.0,0.717391,0.717391,0.737319,0.726449
2,2022-01-01 06:00:00,1,0.630435,2022,1,1,6,0.266506,0.280870,0.000000,0.0,0.672101,0.717391,0.717391,0.737319
3,2022-01-01 07:00:00,1,0.617754,2022,1,1,7,0.269893,0.272716,0.000000,0.0,0.630435,0.672101,0.717391,0.717391
4,2022-01-01 08:00:00,1,0.614130,2022,1,1,8,0.268964,0.278670,0.000000,0.0,0.617754,0.630435,0.672101,0.717391
5,2022-01-01 09:00:00,1,0.583333,2022,1,1,9,0.280928,0.304696,0.080428,0.0,0.614130,0.617754,0.630435,0.672101
6,2022-01-01 10:00:00,1,0.625000,2022,1,1,10,0.360690,0.331155,1.000000,0.0,0.583333,0.614130,0.617754,0.630435
7,2022-01-01 11:00:00,1,0.670290,2022,1,1,11,0.475541,0.250564,1.000000,0.0,0.625000,0.583333,0.614130,0.617754
8,2022-01-01 12:00:00,1,0.737319,2022,1,1,12,0.582208,0.176885,1.000000,0.0,0.670290,0.625000,0.583333,0.614130
9,2022-01-01 13:00:00,1,0.751812,2022,1,1,13,0.589312,0.230964,1.000000,0.0,0.737319,0.670290,0.625000,0.583333


In [11]:
# Use 2024 for testing and 2022-2023 for training
train_data = feature_data[feature_data['year'].isin([2022, 2023])]
test_data = feature_data[feature_data['year'].isin([2024])]

# Drop the timestamp column
train_data.drop(columns=['timestamp'], inplace=True)
test_data.drop(columns=['timestamp'], inplace=True)

# Make station_id categorical
train_data['station_id'] = train_data['station_id'].astype('category')
test_data['station_id'] = test_data['station_id'].astype('category')

FEATURES = ['ctx-1', 'ctx-2', 'ctx-3', 'ctx-4', 'apparent_temperature', 'wind_speed_10m', 'sunshine_duration', 'rain']
TARGET = 'docks_available'

x = train_data[FEATURES].values
y = train_data[TARGET].values

x_test = test_data[FEATURES].values
y_test = test_data[TARGET].values

# Normalize the features
# scaler = MinMaxScaler(feature_range=(0, 1))
# x = scaler.fit_transform(x)
# y = scaler.fit_transform(y.reshape(-1, 1))
# Reshape the input data to be 3D
x = x.reshape((x.shape[0], 1, x.shape[1]))
# Reshape the output data to be 2D
y = y.reshape((y.shape[0], 1))

x_test = x_test.reshape((x_test.shape[0], 1, x_test.shape[1]))
y_test = y_test.reshape((y_test.shape[0], 1))




/tmp/ipykernel_119/3141811407.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data.drop(columns=['timestamp'], inplace=True)
/tmp/ipykernel_119/3141811407.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data.drop(columns=['timestamp'], inplace=True)
/tmp/ipykernel_119/3141811407.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['station_id'] = tra

In [12]:
# Build the LSTM model
# Build the LSTM model
model = Sequential()
model.add(Input(shape=(x.shape[1], x.shape[2])))  # Define input shape using Input layer
model.add(LSTM(50, return_sequences=False, activation='relu'))  # LSTM layer
model.add(Dense(1))  # Output layer
model.compile(loss='mse', optimizer='adam')

# Train the model
model.fit(x, y, epochs=50, batch_size=16384, verbose=2)

# Make predictions
yhat = model.predict(x_test, verbose=2)

# Evaluate the model
train_score = model.evaluate(x, y, verbose=2)
test_score = model.evaluate(x_test, y_test, verbose=2)
print('Train Score: %.2f RMSE' % (train_score**0.5))
print('Test Score: %.2f RMSE' % (test_score**0.5))

Epoch 1/50
206/206 - 8s - 41ms/step - loss: 0.0708
Epoch 2/50
206/206 - 7s - 36ms/step - loss: 0.0146
Epoch 3/50
206/206 - 7s - 36ms/step - loss: 0.0132
Epoch 4/50
206/206 - 7s - 36ms/step - loss: 0.0130
Epoch 5/50
206/206 - 7s - 36ms/step - loss: 0.0129
Epoch 6/50
206/206 - 7s - 36ms/step - loss: 0.0128
Epoch 7/50
206/206 - 7s - 36ms/step - loss: 0.0127
Epoch 8/50
206/206 - 7s - 36ms/step - loss: 0.0126
Epoch 9/50
206/206 - 7s - 36ms/step - loss: 0.0125
Epoch 10/50
206/206 - 7s - 36ms/step - loss: 0.0125
Epoch 11/50
206/206 - 7s - 36ms/step - loss: 0.0124
Epoch 12/50
206/206 - 7s - 36ms/step - loss: 0.0124
Epoch 13/50
206/206 - 7s - 36ms/step - loss: 0.0124
Epoch 14/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 15/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 16/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 17/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 18/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 19/50
206/206 - 7s - 36ms/step - loss: 0.0123
Epoch 20/50
206/206 -